# Cheapest Flights Within K Stops

## Topic
Graph shortest path + Bellman-Ford relaxation with a stop limit.

## Easy Definition
Find the cheapest route from `src` to `dst` when you can use at most `k` intermediate stops.

Number of allowed flights = number of stops + 1.
Therefore, `k` stops means at most `k + 1` edges.

## Core Logic
1. Create `distance[n]` and set `distance[src] = 0`; set every other node to `INT_MAX`.
2. Repeat relaxation `k + 1` times because each pass allows one more flight.
3. Copy `distance` into `temp` before processing all flights.
4. For each directed flight `u -> v` with cost `wt`:
   - Relax only if `u` has a known price.
   - Update `temp[v]` if `distance[u] + wt` is cheaper.
5. Copy `temp` back into `distance` after the complete pass.

Using `temp` prevents one pass from taking multiple flights. This is level-by-level (one flight at a time) relaxation.

If `distance[dst]` remains `INT_MAX`, no valid route exists within `k` stops. Otherwise, return the cheapest price.

## Algorithm
1. Initialize all distances to infinity and the source distance to `0`.
2. Run `k + 1` passes.
3. In every pass:
   - Copy the previous pass's distances to `temp`.
   - Process every flight `(u, v, wt)`.
   - If `distance[u] != INT_MAX` and `distance[u] + wt < temp[v]`, update `temp[v]`.
4. Assign `distance = temp`.
5. Return `distance[dst]`; if it is still infinity, return `-1`.

## Dry Run
Input:
- `n = 3`
- `flights = [[0,1,100], [1,2,100], [0,2,500]]`
- `src = 0`, `dst = 2`, `k = 1`

`distance = [0, INF, INF]`

**Pass 1: at most 1 flight**
- Process `(0,1,100)`:
  - `0 + 100 < INF`, so `temp[1] = 100`.
- Process `(1,2,100)`:
  - `distance[1]` is still `INF`, so skip it.
- Process `(0,2,500)`:
  - `0 + 500 < INF`, so `temp[2] = 500`.
- `distance = [0, 100, 500]`

**Pass 2: at most 2 flights**
- Process `(0,1,100)`:
  - `100 < 100` is false.
- Process `(1,2,100)`:
  - `distance[1] = 100`, so `100 + 100 < temp[2]`.
  - Update `temp[2] = 200`.
- Process `(0,2,500)`:
  - `500 < 200` is false.
- `distance = [0, 100, 200]`

There are at most 2 flights, or 1 stop, so the result is `200`.

## Important Code Point
```cpp
vector<int> temp = distance;
```
This is the key difference from normal Bellman-Ford. It creates a snapshot of distances before the pass, so the same pass cannot chain more than one flight.

## Time Complexity
- Number of passes: `k + 1`
- Flights processed in each pass: `E`
- **Time: O((K + 1) * E)**

## Space Complexity
- `distance`: `O(V)`
- `temp`: `O(V)`
- Extra space: `O(V)`

**Space: O(V)**
```cpp
class Solution {
public:
    int findCheapestPrice(int n, vector<vector<int>>& flights, int src, int dst, int k) {

        vector<int> distance(n, INT_MAX);
        distance[src] = 0;

        for(int i = 0; i <= k; i++) {

            vector<int> temp = distance;

            for(auto &e : flights) {

                int u = e[0];
                int v = e[1];
                int wt = e[2];

                if(distance[u] != INT_MAX &&
                   distance[u] + wt < temp[v]) {

                    temp[v] = distance[u] + wt;
                }
            }

            distance = temp;
        }

        if(distance[dst] == INT_MAX)
            return -1;

        return distance[dst];//TC:O((K+1)*E) || SC :O(V)
    }
};
```